# Lecture 5: Exploratory Data Analysis for Forest Fires

### Short, simple, self-study notes

In the previous lesson, the dataset was cleaned. Now we explore it before training Ridge, Lasso, or Elastic Net.

**Main idea:** EDA helps us understand distributions, class balance, correlations, outliers, and useful patterns before a model is trained.

## 1. Save a cleaned dataset correctly

After cleaning, it is useful to save a reusable CSV. This avoids repeating the same cleaning steps every time.

**Example:** df.to_csv("cleaned_data.csv", index=False)

Use index=False so that pandas does not save the row index as an unwanted extra column. The practical folder already contains a cleaned version of this dataset, so this notebook reads that file.

## 2. Choose the target and prepare a copy

For the regression project, **FWI** is the numeric target. Day, month, and year are not used in the first regression model, but we keep them in the full dataset because month is useful for the fire-pattern analysis.

The Classes column contains fire and not fire. For a numeric analysis, we encode not fire as 0 and fire as 1.

**Important:** First remove extra spaces and make text lowercase. Otherwise fire, fire with a trailing space, and FIRE could be treated as different labels.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

data_path = Path("Ridge Lassso Elastic Regression Practicals") / "Algerian_forest_fires_cleaned_dataset.csv"
df = pd.read_csv(data_path)

df["Classes"] = df["Classes"].str.strip().str.lower()
class_map = {"not fire": 0, "fire": 1}
df["Classes_encoded"] = df["Classes"].map(class_map)

if df["Classes_encoded"].isna().any():
    unknown = df.loc[df["Classes_encoded"].isna(), "Classes"].unique()
    raise ValueError(f"Unexpected class labels: {unknown}")

df_model = df.drop(columns=["day", "month", "year", "Classes"]).copy()
print("Model-data shape after dropping date and text class columns:", df_model.shape)
df[["Classes", "Classes_encoded", "FWI", "Region"]].head()

## 3. Distribution plots

A histogram shows how values are spread. It can reveal skew, a common value range, or unusual values.

- A roughly balanced bell shape is often called approximately normal.
- A long tail to the right is right-skewed.
- A long tail to the left is left-skewed.

Do not remove a value only because it looks unusual. First check whether it is a real observation, a data-entry mistake, or a valid but rare event.

In [ ]:
numeric_columns = df_model.select_dtypes(include="number").columns
df_model[numeric_columns].hist(bins=25, figsize=(15, 11), edgecolor="white")
plt.suptitle("Feature distributions after cleaning", y=1.02, weight="bold")
plt.tight_layout()
plt.show()

## 4. Class balance: fire versus not fire

A class-count plot and pie chart show the proportion of fire and not-fire records. This matters most when building a classification model, but it is still helpful context for the dataset.

A strong class imbalance means that accuracy alone can be misleading. For example, if 95% of rows are not fire, a model that always predicts not fire gets 95% accuracy but is not useful for finding fires.

In [ ]:
class_counts = df["Classes"].value_counts().reindex(["fire", "not fire"])
class_percentages = class_counts / class_counts.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
class_counts.plot(kind="bar", ax=axes[0], color=["#e76f51", "#457b9d"], rot=0)
axes[0].set_title("Number of fire and not-fire rows")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Rows")

axes[1].pie(class_counts, labels=class_counts.index, autopct="%.1f%%",
            colors=["#e76f51", "#457b9d"], startangle=90)
axes[1].set_title("Class percentage")

fig.suptitle("Fire-label balance in the dataset", weight="bold")
fig.tight_layout()
plt.show()

class_percentages.round(2)

## 5. Correlation and multicollinearity

Correlation measures how two numeric variables move together:

- Close to +1: strong positive relationship.
- Close to -1: strong negative relationship.
- Close to 0: weak linear relationship.

The diagonal is always 1 because each feature is perfectly correlated with itself.

**Multicollinearity** means input features are strongly related to one another. It can make ordinary linear-regression coefficients unstable. Ridge and Elastic Net are often useful when this happens because they shrink related feature weights.

In [ ]:
correlation = df_model.corr(numeric_only=True)

plt.figure(figsize=(12, 9))
sns.heatmap(correlation, cmap="coolwarm", center=0, square=True, linewidths=0.3)
plt.title("Correlation heatmap: lighter or darker colors show stronger relationships", weight="bold")
plt.show()

print("Correlation with FWI, from strongest to weakest:")
correlation["FWI"].sort_values(ascending=False).round(3)

## 6. Box plot: a quick outlier check

A box plot shows the median, middle half of the data, spread, and possible outliers.

- The line inside the box is the median.
- The box covers the middle 50% of values.
- Points beyond the whiskers are possible outliers.

Possible outliers are not automatically bad data. In forest-fire data, an unusually high FWI may be a real important event.

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=df["FWI"], color="#70a1d7")
plt.title("FWI box plot: inspect possible outliers", weight="bold")
plt.xlabel("FWI")
plt.show()

## 7. Monthly fire analysis by region

The dataset contains June to September records. We can count fire and not-fire records month by month for each region.

This is why we kept the original full dataframe. If we had permanently removed month before EDA, we could not make this seasonal plot.

In [ ]:
region_names = {0: "Bejaia", 1: "Sidi Bel-Abbes"}
month_names = {6: "Jun", 7: "Jul", 8: "Aug", 9: "Sep"}

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for axis, region_code in zip(axes, [0, 1]):
    region_data = df[df["Region"] == region_code]
    sns.countplot(data=region_data, x="month", hue="Classes",
                  order=[6, 7, 8, 9], hue_order=["fire", "not fire"],
                  palette={"fire": "#e76f51", "not fire": "#457b9d"}, ax=axis)
    axis.set_title(f"{region_names[region_code]} region")
    axis.set_xlabel("Month")
    axis.set_ylabel("Number of rows")
    axis.set_xticklabels([month_names[month] for month in [6, 7, 8, 9]])

fig.suptitle("Monthly fire analysis by region", weight="bold")
fig.tight_layout()
plt.show()

monthly_fire_counts = df[df["Classes"] == "fire"].groupby(["Region", "month"]).size()
print("Month with the most fire rows in each region:")
for region_code, values in monthly_fire_counts.groupby(level=0):
    month = values.droplevel(0).idxmax()
    print(f"{region_names[region_code]}: {month_names[month]}")

### Observation from the monthly plot

The code calculates the busiest fire month for each region instead of guessing it. In this dataset, the counts show the strongest fire activity during the summer months, especially August. September has fewer fire records.

This is an observation about this dataset only. A different place or year can have a different seasonal pattern.

## 8. Final revision card

- Save cleaned data with index=False so no unwanted index column is created.
- Clean text labels before encoding them as numbers.
- Use histograms to understand distributions.
- Use count plots and pie charts to understand class balance.
- Use a correlation heatmap to find relationships and possible multicollinearity.
- Use box plots to inspect possible outliers, not to delete data blindly.
- Keep date columns during EDA when they help answer a question such as monthly fire patterns.
- Next step: split the regression data, scale inside a pipeline, and train the models.

### One-line interview answer

**I perform EDA after cleaning to understand distributions, class balance, outliers, correlations, and feature relationships before selecting and training a model.**